# Background Subtraction

Every column of a long-slit frame holds the night-sky spectrum along the whole slit, with the
object's spectrum on top of it in a handful of rows. The sky is neither a small correction nor a
constant one: in a five-pixel aperture on this frame it is about 2 percent of the flux on average,
but it ranges from under 1 percent to nearly 10 percent, because the sky has bright emission lines
where the star does not.

`specreduce.background.Background` estimates the sky from windows of the slit beside the source
and subtracts it. It models the sky in each column as a single number, so whatever falls inside
the windows is averaged into that number, and most of the care in using it goes into placing the
windows. This chapter works on the frame rectified in [Chapter 6](06-tilt-correction.ipynb) and
traced as in [Chapter 4](04-tracing.ipynb), and ends with the window settings the rest of the book
uses.

In [ ]:
import sys

sys.path.append("../src")

import numpy as np
import matplotlib.pyplot as plt
from astropy.modeling import models
from astropy.visualization import simple_norm
from specreduce.background import Background
from specreduce.tilt_solution import TiltSolution
from specreduce.tracing import FitTrace

from common import TILT_SOLUTION_FILE, fit_tilt_solution, read_data

plt.rc("figure", figsize=(8, 2.4), dpi=110)
plt.rc("font", size=9)

arcs, lamps, obj = read_data()
ny, nx = obj.data.shape
rows = np.arange(ny)
columns = np.arange(nx)

Two things have to be in place before we can measure the sky. The frame is **rectified**, because the windows sit tens of
rows from the source and their columns must hold the same wavelengths as the source's. The tilt
solution Chapter 6 wrote is read back, and the guard refits it if the file is missing, so the
chapter runs on its own. The source is then **traced** on the rectified frame with Chapter 4's
settings.

In [ ]:
if not TILT_SOLUTION_FILE.exists():  # normally written by Chapter 6
    fit_tilt_solution(arcs).to_asdf(TILT_SOLUTION_FILE)

solution = TiltSolution.from_asdf(TILT_SOLUTION_FILE)

image = solution.resample(obj)

trace = FitTrace(
    image, bins=32, peak_method="gaussian", trace_model=models.Chebyshev1D(4)
)
trace_y = np.asarray(trace.trace)

## Reading the slit

Before placing any window, we look at what is on the slit. The median along the dispersion axis
gives the cross-dispersion profile, and this time the interest is in everything except the
target.

In [ ]:
slit = np.median(image.data, axis=1)

fig, ax = plt.subplots(figsize=(8, 3.0), constrained_layout=True)
ax.plot(rows, slit, lw=0.8)

for row, name in (
    (135, "target"),
    (117, "faint star"),
    (36, "faint star"),
    (311, "faint star"),
):
    ax.annotate(
        name,
        (row, slit[row]),
        (row, slit[row] * 1.5),
        size=8,
        ha="center",
        arrowprops=dict(arrowstyle="-", lw=0.6, color="0.4"),
    )

ax.set(
    yscale="log",
    xlim=(0, ny - 1),
    ylim=(30, 2e5),
    xlabel="Cross-dispersion [pix]",
    ylabel="Median signal [e$^-$]",
);

The sky is flat, near 640 e⁻ along the whole slit, so a window can go wherever the sources allow.
The sources are the constraint. Besides the star at row 311 that Chapter 4 set aside, two faint
ones sit at rows **117** and **36**, easy to miss on a first look and bright enough to poison a
sky estimate, and the one at 117 is only 18 rows from the target. The target itself has wings:
ten rows from the trace it still adds about half the sky level, less than 1 percent of its peak,
and it does not fall below 1 percent of the sky until 40 to 60 rows out. Windows must therefore
be **far enough** from the trace to escape its wings and **near enough** to stay on clean sky
between the other sources.

## How `Background` works

A `Background` is defined by one or more **windows**, strips parallel to a trace and displaced
from it across the slit, each set by a `separation` (the offset of its center from the trace) and
a `width`. Because they follow the trace, the windows stay at the same distance from the source as
it curves; they are the `trace + delta` of Chapter 4. `Background.two_sided(image, trace,
separation, width=...)` places one window on each side, at ±`separation`, and
`Background.one_sided` places one, on the side given by the sign of `separation`.

In each column the pixels inside the windows are reduced to **one number**, an average or a
median, which becomes the sky for every row of that column. The sky model across the slit is a
constant: it cannot follow a gradient or leave out a source that falls in a window.

A `Background` computes the sky when it is created and returns it through three methods.
`bkg_spectrum()` gives the per-column sky as a 1D spectrum along the dispersion axis, `bkg_image()`
tiles that spectrum into a 2D image of the frame's shape, and `sub_image()` returns the frame minus
that image, the background-subtracted frame that extraction works on.

In [ ]:
separation, width = 60, 20
bkg = Background.two_sided(image, trace, separation=separation, width=width)

bkg_image = bkg.bkg_image()
sub_image = bkg.sub_image()
bkg_spectrum = bkg.bkg_spectrum()

print(f"bkg_image     {bkg_image.flux.shape}  {bkg_image.flux.unit}")
print(
    f"sub_image     {sub_image.flux.shape}  uncertainty: "
    f"{type(sub_image.uncertainty).__name__}"
)
print(
    f"bkg_spectrum  {bkg_spectrum.flux.shape}  uncertainty: "
    f"{type(bkg_spectrum.uncertainty).__name__}"
)

All three products are `specutils` `Spectrum` objects that carry uncertainties: `Background`
propagates the input variance through the averaging, which Chapter 8's optimal extraction needs
for its weights. The window geometry and the result:

In [ ]:
def show_windows(ax, separation, width, ylim=(10, 260), vmax=250_000):
    """Draw the frame with the trace and the two background windows marked."""
    ax.imshow(
        image.data,
        origin="lower",
        aspect="auto",
        cmap=plt.cm.Blues,
        norm=simple_norm(image.data, stretch="log", vmin=0, vmax=vmax),
    )
    ax.plot(trace_y, c="C1", lw=1.0)
    for sign in (1, -1):
        center = trace_y + sign * separation
        ax.fill_between(
            columns,
            center - width / 2,
            center + width / 2,
            color="C3",
            alpha=0.25,
            lw=0,
        )
        ax.plot(center, c="C3", lw=0.6, ls="--")
    ax.set(xlim=(0, nx - 1), ylim=ylim, ylabel="Cross-dispersion [pix]")


fig, axs = plt.subplots(3, figsize=(8, 5.4), sharex="all", constrained_layout=True)

show_windows(axs[0], separation, width)
axs[0].set_title(
    f"rectified frame, windows at $\\pm${separation} px, width {width}", size=9
)

for ax, panel, label in (
    (axs[1], bkg_image.flux.value, "background model"),
    (axs[2], sub_image.flux.value, "background subtracted"),
):
    ax.imshow(
        panel,
        origin="lower",
        aspect="auto",
        cmap=plt.cm.Blues,
        norm=simple_norm(panel, stretch="log", vmin=0, vmax=250_000),
    )
    ax.text(0.01, 0.9, label, transform=ax.transAxes, va="top", size=8)
    ax.set(ylim=(10, 260), ylabel="Cross-dispersion [pix]")

axs[-1].set_xlabel("Dispersion [pix]");

The background model in the middle panel, `bkg_image()`, is one constant per column: banded
vertically, varying only along the dispersion axis, with the sky emission lines as bright
stripes. In the subtracted frame below it, `sub_image()`, the stripes are gone and the glow around
the trace has flattened out.

The background spectrum on its own, `bkg_spectrum()`, is the sky as `Background` sees it:

In [ ]:
fig, ax = plt.subplots(figsize=(8, 2.6), constrained_layout=True)
ax.plot(bkg_spectrum.flux.value, lw=0.7)
ax.set(xlim=(0, nx - 1), xlabel="Dispersion [pix]", ylabel="Sky [e$^-$ pix$^{-1}$]");

It is a recognizable night-sky spectrum, a smooth continuum with strong, narrow emission lines.
The brightest, near column 743, is about 20 times the faintest point, which is why the
contamination is so strongly wavelength dependent.

## Choosing the separation

Too close, the windows catch the target's wings; too far, its neighbors. The optimum can be found
empirically: subtract the background, then take the median residual in a strip of blank sky just
outside the upper window, which should be zero if the sky estimate is right.

In [ ]:
def residual_outside(separation, width, gap=5, depth=20):
    """Median residual in a blank strip just beyond the upper background window."""
    background = Background.two_sided(image, trace, separation=separation, width=width)
    subtracted = background.sub_image().flux.value
    low = int(round(trace_y.mean() + separation + width / 2 + gap))
    return np.median(np.median(subtracted[low : low + depth], axis=0))


separations = np.arange(25, 106, 10)
residuals = np.array([residual_outside(s, 20) for s in separations])

for s, r in zip(separations, residuals):
    print(f"separation {s:3d}:  residual {r:8.2f} e-")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 2.8), constrained_layout=True)
ax.plot(separations, residuals, "o-", lw=0.9, ms=4)
ax.axhline(0, c="0.6", lw=0.8, ls="--")
ax.annotate(
    "source wings\nin the window",
    (25, residuals[0]),
    (40, -70),
    size=8,
    arrowprops=dict(arrowstyle="->", lw=0.6, color="0.4"),
)
ax.annotate(
    "window reaches\nthe trace at row 36",
    (95, residuals[-1]),
    (62, -45),
    size=8,
    arrowprops=dict(arrowstyle="->", lw=0.6, color="0.4"),
)
ax.set(xlabel="Separation [pix]", ylabel="Residual sky [e$^-$]");

The curve has the shape the slit profile predicts. At separation 25 the windows sit on the target's
wings, the sky is overestimated, and the rest of the frame is driven 81 e⁻ negative, over 10
percent of the sky level. From 55 to 85 the residual stays within about 2 e⁻ of zero, and at 95
it collapses to −37 e⁻, because the lower window then spans rows 30 to 50 and takes in the star
at row 36. Neither failure produces a warning. We adopt **separation 60, width 20**, clear of the
wings and of all three neighboring sources, and the later chapters reuse it.

:::{note}
The `width` trades noise against risk. A wider window averages more pixels, but spans more of the
slit and is more likely to reach a neighbor or a change in illumination. Two 20-row windows give
40 pixels per column, enough that the sky estimate adds negligible noise to the extraction.
:::

## One-sided and two-sided windows

`two_sided` averages a window on each side of the trace; `one_sided` uses one, on the side the
sign of `separation` selects. At the adopted settings:

In [ ]:
options = (
    ("two_sided", Background.two_sided(image, trace, separation=60, width=20)),
    ("one_sided, above", Background.one_sided(image, trace, separation=60, width=20)),
    ("one_sided, below", Background.one_sided(image, trace, separation=-60, width=20)),
)

for label, background in options:
    flux = background.bkg_spectrum().flux.value
    print(f"{label:>18}:  sky at column 512 = {flux[512]:7.2f} e-")

The two sides disagree by about 3 percent, 800 e⁻ above the trace against 823 e⁻ below, and the
two-sided value, 811 e⁻, falls between them: the sky has a real gradient across the slit. A
constant cannot follow it, but two windows placed symmetrically about the trace evaluate it at the
trace, where a linear gradient cancels exactly, while a one-sided estimate is biased by half the
gradient across the separation. Use `one_sided` only when one side is unusable (for example, due to
a neighbor, a bad column, or the edge of the slit).

## `statistic` and outlier rejection

`statistic` chooses between `"average"` (the default, which weights partial pixels at the window
edges) and `"median"`, and `sigma` sigma-clips each column before combining, with a default of 5.0
(`sigma=None` disables it).

In [ ]:
for statistic in ("average", "median"):
    flux = (
        Background.two_sided(image, trace, separation=60, width=20, statistic=statistic)
        .bkg_spectrum()
        .flux.value
    )
    print(f"{statistic:>8}: sky at column 512 = {flux[512]:7.2f} e-")

The two statistics agree to about 1 e⁻: with 40 well-behaved pixels per column, the mean and the
median measure the same thing. With clipping on, `average` is the better choice, being less noisy
than the median on well-behaved data and handling fractional pixels at the window edges.

Clipping changes the sky in **only seven of the 1000 columns**, because an average over 40 pixels
leaves ordinary noise nothing to reject. The seven columns are three **cosmic rays** that landed
in the windows, each spread over two or three neighboring columns by the resampling of Chapter 6,
and clipping lowers their sky by 14 to 52 e⁻; a dead pixel would pull the other way and be treated
alike. `sigma` is therefore a no-op almost everywhere and a correction of tens of electrons in a
few unpredictable places, and it costs nothing, so we keep the default. It cleans only the
windows, not a cosmic ray on the trace itself, which is `astroscrappy`'s job before any of this.
It also assumes that the pixels in a window share one distribution, so on a steep illumination
gradient or an unrectified frame an aggressive `sigma` can reject real signal; do not lower the
default without checking.

## Summary

- `Background` estimates the sky in windows **parallel to the trace**, offset by `separation` and
  `width` rows wide, and reduces each column to one number, so anything left in a window is
  averaged into the sky without a warning.
- **Read the slit profile before placing windows.** This frame carries four sources (rows 36, 117,
  135 and 311), and the target's wings reach 40 to 60 rows out.
- Separation has an optimum. At 25 rows the target's wings inflate the sky and drive the frame
  81 e⁻ negative; at 95 the lower window takes in the star at row 36. From 55 to 85 the residual
  stays within about 2 e⁻ of zero.
- Prefer `two_sided`. The sky differs by about 3 percent between the two sides here, and
  symmetric windows cancel the linear gradient that biases a one-sided estimate.
- `statistic="average"` with the default `sigma=5.0` is the right choice. Clipping acts in only
  seven of 1000 columns, correcting cosmic rays in the windows by up to 52 e⁻, and does nothing
  about a hit on the trace.

Next: [Spectral Extraction](08-extraction.ipynb), which collapses the background-subtracted frame
to one dimension.